In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
from benchmark import port_and_benchmark, benchmark_models

In [3]:
load_dotenv(override=True)
openrouter_base_url, openrouter_api_key = os.getenv('OPENROUTER_BASE_URL'), os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"Openrouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("Openrouter API Key not set")

Openrouter API Key exists and begins sk-or-v1


In [4]:
# Connect to client libraries
openrouter = OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key)

In [5]:
OPENAI_MODEL = "openai/gpt-5-nano"
CLAUDE_MODEL = "anthropic/claude-3.5-haiku"
GROK_MODEL = "x-ai/grok-4.1-fast"
GEMINI_MODEL = "google/gemini-2.5-flash-lite-preview-09-2025"

In [6]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '12th Gen Intel(R) Core(TM) i9-12900H',
  'cores_logical': 20,
  'cores_physical': 14,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'g++': 'g++.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [7]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [8]:
compile_command = ["g++", "-std=c++17", "-O3", "main.cpp", "-o", "main.exe"]
run_command = ["./main"]

In [9]:
from benchmark import benchmark_models

models = {
    "GPT-5 Nano":       OPENAI_MODEL,
    "Claude 3.5 Haiku": CLAUDE_MODEL,
    "Grok 4.1 Fast":    GROK_MODEL,
    "Gemini 2.5 Flash": GEMINI_MODEL,
}

results = benchmark_models(
    openrouter, models, pi,
    compile_command, run_command,
    system_info=system_info
)


🐍 Running Python baseline...
Result: 3.141592656089
Execution Time: 95.853235 seconds
   Python time: 95.853235s

🔄 Porting with openai/gpt-5-nano...
   TTFT:       133.25s
   Generation: 133.49s
   File size:  867 bytes
⚙️  Compiling & running 3x...
✅ openai/gpt-5-nano
   Run 1: 1.305619s
   Run 2: 1.287392s
   Run 3: 1.287194s
   Avg:   1.293402s

🔄 Porting with anthropic/claude-3.5-haiku...
   TTFT:       1.90s
   Generation: 5.15s
   File size:  837 bytes
⚙️  Compiling & running 3x...
✅ anthropic/claude-3.5-haiku
   Run 1: 1.243624s
   Run 2: 1.294883s
   Run 3: 1.279758s
   Avg:   1.272755s

🔄 Porting with x-ai/grok-4.1-fast...
   TTFT:       46.76s
   Generation: 47.69s
   File size:  705 bytes
⚙️  Compiling & running 3x...
✅ x-ai/grok-4.1-fast
   Run 1: 1.233298s
   Run 2: 1.233978s
   Run 3: 1.235738s
   Avg:   1.234338s

🔄 Porting with google/gemini-2.5-flash-lite-preview-09-2025...
   TTFT:       15.74s
   Generation: 18.54s
   File size:  2134 bytes
⚙️  Compiling & running 3